In [1]:
import pandas as pd
import numpy as np

1. Реализация PWM "с нуля"(1 балл)

Цель: понять математику под капотом, не полагаясь на библиотеки.

Задачи:
1. Напишите функцию на numpy, которая возвращает PFM (Position Frequency Matrix).
2. Реализуйте переход PFM → PPM с псевдосчётом α = 0.1.
3. Реализуйте переход PPM → PWM. Фон генома: P (A) = P (T ) = 0.295, P (G) = P (C) = 0.205.
4. Найдите максимальный и минимальный теоретически возможный скор для вашей PWM. Какие последовательности их дают?

Ответ: Код функций перевода PFM → PPM → PWM. Вывод получившихся матриц. Найденные макси-
мальный/минимальный скоры и соответствующие им последовательности.

In [2]:
sites = [ " GAGGTAAAC " , " TCCGTAAGC " , " CAGGTTGGA " ,
" ACAGTCAGC " , " TAGGTCAGC " , " CAGGTCAGC " ,
" CAGGTCGAT " , " CAGGTCAGC " , " CAGGTCAGC " ,
" CAGGTTGGC " ]

In [10]:
def pfm(sequences_list):
    l = len(sequences_list[0])
    bases = 'ACGT'
    pfm = np.zeros((4, l), dtype = float)
    for seq in sequences_list:
        for j, base in enumerate(seq):
            if base in bases:
                i = bases.index(base)
                pfm[i, j] += 1
    return pfm

In [11]:
pfm(sites)

array([[ 0.,  1.,  8.,  1.,  0.,  0.,  2.,  7.,  2.,  1.,  0.],
       [ 0.,  6.,  2.,  1.,  0.,  0.,  6.,  0.,  0.,  8.,  0.],
       [ 0.,  1.,  0.,  8., 10.,  0.,  0.,  3.,  8.,  0.,  0.],
       [ 0.,  2.,  0.,  0.,  0., 10.,  2.,  0.,  0.,  1.,  0.]])

In [12]:
def pfm_to_ppm(pfm, alpha):
    ppm = pfm + alpha
    return ppm / ppm.sum(axis = 0)

In [16]:
pfm_to_ppm(pfm(sequences_list = sites), alpha = 0.1)

array([[0.25      , 0.10576923, 0.77884615, 0.10576923, 0.00961538,
        0.00961538, 0.20192308, 0.68269231, 0.20192308, 0.10576923,
        0.25      ],
       [0.25      , 0.58653846, 0.20192308, 0.10576923, 0.00961538,
        0.00961538, 0.58653846, 0.00961538, 0.00961538, 0.77884615,
        0.25      ],
       [0.25      , 0.10576923, 0.00961538, 0.77884615, 0.97115385,
        0.00961538, 0.00961538, 0.29807692, 0.77884615, 0.00961538,
        0.25      ],
       [0.25      , 0.20192308, 0.00961538, 0.00961538, 0.00961538,
        0.97115385, 0.20192308, 0.00961538, 0.00961538, 0.10576923,
        0.25      ]])

In [22]:
def ppm_to_pwm(ppm, A_background, C_background, G_background, T_background):
    background = np.array([A_background, C_background, G_background, T_background])[:, np.newaxis]
    pwm = np.log2(ppm / background)
    return pwm

In [23]:
ppm_to_pwm(pfm_to_ppm(pfm(sequences_list = sites), alpha = 0.1), 0.295, 0.205, 0.205, 0.295)

array([[-0.23878686, -1.47979496,  1.40062343, -1.47979496, -4.93922658,
        -4.93922658, -0.54690915,  1.21052054, -0.54690915, -1.47979496,
        -0.23878686],
       [ 0.28630419,  1.5166018 , -0.02181811, -0.95470391, -4.41413553,
        -4.41413553,  1.5166018 , -4.41413553, -4.41413553,  1.92571447,
         0.28630419],
       [ 0.28630419, -0.95470391, -4.41413553,  1.92571447,  2.24407595,
        -4.41413553, -4.41413553,  0.54006078,  1.92571447, -4.41413553,
         0.28630419],
       [-0.23878686, -0.54690915, -4.93922658, -4.93922658, -4.93922658,
         1.71898491, -0.54690915, -4.93922658, -4.93922658, -1.47979496,
        -0.23878686]])

In [24]:
def pwm_scores(pwm):
    L = pwm.shape[1]
    max_scores = np.max(pwm, axis=0)
    min_scores = np.min(pwm, axis=0)
    
    max_score = np.sum(max_scores)
    min_score = np.sum(min_scores)
    
    bases = 'ACGT'
    max_seq = ''.join(bases[np.argmax(pwm[:,j])] for j in range(L))
    min_seq = ''.join(bases[np.argmin(pwm[:,j])] for j in range(L))
    
    return max_score, max_seq, min_score, min_seq

In [25]:
pwm_scores(ppm_to_pwm(pfm_to_ppm(pfm(sequences_list = sites), alpha = 0.1), 0.295, 0.205, 0.205, 0.295))

(15.957160210894493, 'CCAGGTCAGCC', -40.42099921060329, 'AATTAAGTTGA')